# Assignment of survey data to a command unit
The survey was done on a governorate level. Therefore we do know from which governorate each survey point was taken, but we do not have this information on the command unit level. Since our study is focused on the command unit level, we need to assign the survey data to one of the command units, based on the available data on the governorates. 

One simple approach to assign the data to command units would be to make use of the shapefiles for both the command units and the governorates and distribute the survey points based on the percentage of an area in a governorate that belongs to each command unit. However, this does not take into account the distribution of the population within the governorates. A large percentage of a governorate might have an overlap with one command unit, but could be largely uninhabited, making it unlikely that a large percentage of the survey points were taken from that command unit.

Instead, we assign the survey points to command units based on the distribution of the rural population in the governorate. For each governorate we sum the total rural population living in the area. We then overlay the population raster with the command units. For each of the command units, we then again sum the rural population that is in both the governorate and the command unit. From this we get a percentage of the rural population that is living in each command unit. This results in a distribution we can use to randomly assign command units to the survey points.

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio

from pathlib import Path
from exactextract import exact_extract
from shapely.geometry import Polygon, MultiPolygon, GeometryCollection

## 1. Preprocessing the survey data
First we preprocess the survey data to combine the individual data with the houshold data, following the steps in the notebook: '01_preprocess_lfs2022_individual'

In [2]:
REPO_ROOT = Path.cwd().parent
LFS_DTA = (
    REPO_ROOT
    / "ERF_Data/Data"
    / "Labor Force Survey, LFS 2022 - Egypt, Arab Rep., 2022"
    / "Egypt 2022-LFS IND-V1.dta"
)
HH_DTA = (
    REPO_ROOT
    / "ERF_Data/Data"
    / "Labor Force Survey, LFS 2022 - Egypt, Arab Rep., 2022"
    / "Egypt 2022-LFS HH-V1.dta"
)
CROSSWALK_CSV = (
    REPO_ROOT / "ERF_Data/Data" / "crosswalk" / "reg_governorate_crosswalk.csv"
)
COMMAND_AREA_CROSSWALK_CSV = (
    REPO_ROOT / "ERF_Data/Data" / "crosswalk" / "command_area_governorate_crosswalk.csv"
)
SPATIAL_XLSX = REPO_ROOT / "Data" / "preprocessed_data_explain.xlsx"
OUTPUT_CSV = Path("lfs2022_individual_preprocessed.csv")

WEEKS_PER_MONTH = 365.25 / 7 / 12  # ~4.348
MIN_COMMAND_AREA_COVERAGE = 0.05
MIN_WORKING_AGE = 15

IND_LABELS = {
    10: "Agriculture, forestry and fishing",
    20: "Mining and quarrying",
    30: "Manufacturing",
    40: "Electricity, gas and water supply",
    50: "Construction",
    60: "Wholesale and retail trade",
    70: "Transportation and storage",
    80: "Accommodation and food service activities",
    90: "Information and communication",
    100: "Financial and insurance activities",
    110: "Real estate, professional and support service activities",
    120: "Public administration and defense",
    130: "Education",
    140: "Human health and social work activities",
    150: "Other activities",
}
EMPS_LABELS = {
    1: "Employee",
    2: "Employer",
    3: "Own-account/self-employed",
    4: "Unpaid family worker",
    5: "Producers cooperative",
    6: "Not classifiable",
}
EDUC_LABELS = {
    1: "None",
    2: "Primary/Lower secondary",
    3: "Secondary",
    4: "Post secondary or equivalent",
    5: "University",
    6: "Postgraduate",
}
EMPSTAB_LABELS = {
    1: "Full time/Regular",
    2: "Part time/Temporary",
    3: "Seasonal/Irregular",
}
REL_LABELS = {
    1: "Head",
    2: "Spouse",
    3: "Son/daughter",
    4: "Parent",
    5: "Sibling",
    6: "Grandchild",
    7: "Other relative",
    8: "Non-relative",
}
OCC_LABELS = {
    10: "Managers",
    20: "Professionals",
    30: "Technicians",
    40: "Clerks",
    50: "Service/sales",
    60: "Skilled agricultural/fishery",
    70: "Craft workers",
    80: "Plant/machine operators",
    90: "Elementary occupations",
    100: "Armed forces",
    998: "Other",
    999: "Not stated",
}
# Standard ERF-harmonized marital-status scheme (code 1 confirmed as "Never married" in the
# dictionary; 2-5 follow the standard ILO/ERF ordering, not independently re-verified against a
# complete codebook; the source dictionary's value list is truncated for this variable).
MART_LABELS = {
    1: "Never married",
    2: "Married",
    3: "Widowed",
    4: "Divorced",
    5: "Separated",
}
YES_NO = {0: "No", 1: "Yes"}

In [3]:
df = pd.read_stata(LFS_DTA, convert_categoricals=False)
hh = pd.read_stata(HH_DTA, convert_categoricals=False)
print("individual shape:", df.shape)
print("household shape:", hh.shape)

individual shape: (299423, 106)
household shape: (76976, 116)


In [4]:
household_size = df.groupby("caseser").size()
n_employed_in_hh = df[df["emps"].notna()].groupby("caseser").size()
head_info = (
    df[df["rel"] == 1].drop_duplicates("caseser").set_index("caseser")[["educ", "sex"]]
)
head_info.columns = ["head_educ", "head_sex"]

print("household_size: computed for", len(household_size), "households")
print(
    "n_employed_in_hh: computed for",
    len(n_employed_in_hh),
    "households (that have >=1 employed member)",
)
print("head_info: found a head for", len(head_info), "households")
print(
    "hh file: malinlf/feminlf/occhd available for",
    hh["caseser"].nunique(),
    "households",
)

household_size: computed for 76976 households
n_employed_in_hh: computed for 57695 households (that have >=1 employed member)
head_info: found a head for 76976 households
hh file: malinlf/feminlf/occhd available for 76976 households


In [5]:
avg_hours_by_ind = df[df["hrswk"] > 0].groupby("ind")["hrswk"].mean()

head_hours = df[df["rel"] == 1].drop_duplicates("caseser").set_index("caseser")["hrswk"]
spouses_ind = df[df["rel"] == 2].sort_values(["caseser", "pnum"]).copy()
spouses_ind["spouse_order"] = spouses_ind.groupby("caseser").cumcount() + 1
spouse_hours = {
    n: spouses_ind[spouses_ind["spouse_order"] == n].set_index("caseser")["hrswk"]
    for n in [1, 2, 3]
}


def convert_irregular(daily_wage, ind_code, real_hours, label):
    used_real = real_hours.notna() & (real_hours > 0)
    hours = real_hours.where(used_real, ind_code.map(avg_hours_by_ind))
    n_earners = (daily_wage > 0).sum()
    n_real = (used_real & (daily_wage > 0)).sum()
    if n_earners:
        print(
            f"  {label}: {n_real}/{n_earners} irregular earners matched to real individual hours "
            f"({n_real/n_earners*100:.1f}%), rest used industry average"
        )
    workdays_per_week = hours / 8
    return daily_wage * workdays_per_week * 4


hh_wage = hh[hh["rururb"] == 0].copy()
hh_wage["head_hrswk"] = hh_wage["caseser"].map(head_hours)
print(
    "Irregular-wage-to-monthly conversion, real-hours match rate (rural households only):"
)
hh_wage["irrwagehd"] = convert_irregular(
    hh_wage["irrgwaghd"].fillna(0), hh_wage["indhd"], hh_wage["head_hrswk"], "head"
)
for n in [1, 2, 3]:
    hh_wage[f"sp_{n}_hrswk"] = hh_wage["caseser"].map(spouse_hours[n])
    hh_wage[f"irrwagesp_{n}"] = convert_irregular(
        hh_wage[f"irrgwagsp_{n}"].fillna(0),
        hh_wage[f"indsp_{n}"],
        hh_wage[f"sp_{n}_hrswk"],
        f"spouse_{n}",
    )

hh_wage["wagehd"] = hh_wage[["empinchd", "sempinchd", "totwaghd", "irrwagehd"]].sum(
    axis=1, skipna=True
)
sp_cols = []
for n in [1, 2, 3]:
    sp_cols += [f"empincsp_{n}", f"sempincsp_{n}", f"totwagsp_{n}", f"irrwagesp_{n}"]
hh_wage["wagesp"] = hh_wage[sp_cols].sum(axis=1, skipna=True)
hh_wage["household_wage_total"] = hh_wage["wagehd"] + hh_wage["wagesp"]
hh_wage["wage_ratio"] = (hh_wage["wagesp"] / hh_wage["household_wage_total"]).fillna(
    0.5
)

print()
print(hh_wage[["household_wage_total", "wage_ratio"]].describe())

Irregular-wage-to-monthly conversion, real-hours match rate (rural households only):
  head: 7511/7538 irregular earners matched to real individual hours (99.6%), rest used industry average
  spouse_1: 195/200 irregular earners matched to real individual hours (97.5%), rest used industry average

       household_wage_total    wage_ratio
count          45426.000000  45426.000000
mean            2209.675539      0.190799
std             3359.425282      0.252778
min                0.000000      0.000000
25%                0.000000      0.000000
50%             2250.000000      0.000000
75%             3200.000000      0.500000
max           150000.000000      1.000000


In [6]:
n_underage = (df["age"] < MIN_WORKING_AGE).sum()
working_age = df[df["age"] >= MIN_WORKING_AGE].copy()
print(
    f"Dropped {n_underage} respondents under age {MIN_WORKING_AGE} (working-age floor)"
)

rural = working_age[working_age["rururb"] == 0].copy()
print("shape after rural filter:", rural.shape)

employed = rural[rural["emps"].notna()].copy()
print("shape after employed filter:", employed.shape)
print(employed["emps"].map(EMPS_LABELS).value_counts())

agri = employed[employed["ind"] == 10].copy()
print("shape after agriculture filter (ind==10):", agri.shape)
print(f"({(agri['sex']==1).sum()} male, {(agri['sex']==2).sum()} female)")

Dropped 100476 respondents under age 15 (working-age floor)
shape after rural filter: (116154, 106)
shape after employed filter: (46027, 106)
emps
Employee                     32521
Own-account/self-employed     9375
Unpaid family worker          3068
Employer                      1032
Name: count, dtype: int64
shape after agriculture filter (ind==10): (14043, 106)
(12019 male, 2024 female)


In [7]:
agri["household_size"] = agri["caseser"].map(household_size)
agri["n_other_employed"] = agri["caseser"].map(n_employed_in_hh) - 1  # exclude self
agri = agri.merge(head_info, on="caseser", how="left")
agri = agri.merge(
    hh[["caseser", "malinlf", "feminlf", "occhd"]], on="caseser", how="left"
)
agri = agri.merge(
    hh_wage[["caseser", "household_wage_total", "wage_ratio"]], on="caseser", how="left"
)

agri[
    [
        "caseser",
        "rel",
        "household_size",
        "n_other_employed",
        "head_educ",
        "head_sex",
        "malinlf",
        "feminlf",
        "occhd",
        "household_wage_total",
        "wage_ratio",
    ]
].sample(5, random_state=1)

,caseser,rel,household_size,n_other_employed,head_educ,head_sex,malinlf,feminlf,occhd,household_wage_total,wage_ratio
272,1.220162e+10,1,4,1,1.0,1,2,0,60.0,2000.0,0.0
4300,1.505043e+11,1,2,1,1.0,1,1,1,10.0,1300.0,0.0
9465,2.304172e+11,1,5,0,2.0,1,1,0,60.0,5100.0,0.0
1060,2.112303e+10,1,5,0,1.0,1,1,0,60.0,2500.0,0.0
588,1.513043e+10,2,7,1,2.0,1,1,1,60.0,1000.0,0.0


In [8]:
valid_hours = agri["hrswk"].notna() & (agri["hrswk"] > 0)
n_dropped = (~valid_hours).sum()
final = agri[valid_hours].copy()
print(f"Dropped {n_dropped} people with missing/zero hours worked")
print(
    f"Remaining: {len(final)} ({(final['sex']==1).sum()} male, {(final['sex']==2).sum()} female)"
)

Dropped 37 people with missing/zero hours worked
Remaining: 14006 (11993 male, 2013 female)


In [9]:
overlap = ((final["totwag"] > 0) & (final["irrgwag"] > 0)).sum()
assert (
    overlap == 0
), f"{overlap} respondents have both totwag and irrgwag > 0, unexpected"

is_regular = final["totwag"] > 0
is_irregular = final["irrgwag"] > 0
is_wage_earner = is_regular | is_irregular

final["wage_type"] = np.select(
    [is_regular, is_irregular], ["regular", "irregular"], default=None
)
final["hourly_wage"] = np.where(
    is_regular,
    final["totwag"] / (final["hrswk"] * WEEKS_PER_MONTH),
    np.where(is_irregular, final["irrgwag"] / (final["hrswk"] / 7), np.nan),
)

print(f"Full population (primary, hours): {len(final)}")
print(
    f"Wage-earning subset (secondary, wage): {is_wage_earner.sum()} "
    f"({(is_wage_earner & (final['sex']==2)).sum()} female)"
)

Full population (primary, hours): 14006
Wage-earning subset (secondary, wage): 7152 (363 female)


In [10]:
final["sex_label"] = final["sex"].map({1: "Male", 2: "Female"})
final["educ_label"] = final["educ"].map(EDUC_LABELS)
final["ind_label"] = final["ind"].map(IND_LABELS)
final["emps_label"] = final["emps"].map(EMPS_LABELS)
final["mart_label"] = final["mart"].map(MART_LABELS)
final["empstab_label"] = final["empstab"].map(EMPSTAB_LABELS)
final["rel_label"] = final["rel"].map(REL_LABELS)
final["occhd_label"] = final["occhd"].map(OCC_LABELS)
final["lit_label"] = final["lit"].map(YES_NO)
final["hlthins_label"] = final["hlthins"].map(YES_NO)
final["socsec_label"] = final["socsec"].map(YES_NO)
final["is_unpaid_family"] = final["emps"] == 4
final["is_wage_employee"] = final["emps"] == 1

In [11]:
crosswalk = pd.read_csv(CROSSWALK_CSV)

merged = final.merge(
    crosswalk[["reg_code", "OBJECTID", "NAME1_"]],
    left_on="reg",
    right_on="reg_code",
    how="left",
)

## 2. Create population distribution matrix
We create a population distribution matrix, that maps the rural population distribution in each governorate to each command unit. We use the shapefiles for the governorates and the command units and a raster file of the rural population in Egypt. Some governorates have a (large) part of the population living in none of the command units. In this case, the percentage that does not live in any of the command units is assigned to 'other'. The distribution matrix is constructed as follows.

<ol>
  <li>Loop over the governorates</li>
  <li>Sum the population living in each governorate, by making an intersection between the governorate polygon and the rural population raster</li>
  <li>For each command unit, make an intersection with the governorate</li>
  <li>For each intersection, make an intersection with the rural population raster and sum the population</li>
  <li>Compute the percentage of the population from the governorate living in the command unit</li>
</ol>

In [12]:
src_dir = (
    Path("~").expanduser()
    / "OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt/04_Data/2026_data/"
)
excel_path = (
    Path("~").expanduser()
    / "OneDrive - Stichting Deltares/Tiaravanni Hermawan's files - Egypt_ERF_data/data_correlation.xlsx"
)

command_gdf = gpd.read_file(src_dir / "Final2_Command_Area.shp")
conversion_df = pd.read_excel(excel_path, sheet_name="command_area")

mapping = command_gdf.merge(
    conversion_df[["area_map_name", "area_name"]],
    left_on="OBJECTID",
    right_on="area_map_name",
).set_index("Name")["area_name"]

In [13]:
def to_multipolygon(geom):
    if geom.geom_type == "Polygon":
        return MultiPolygon([geom])
    return geom


def keep_polygons(geom):
    if isinstance(geom, GeometryCollection):
        polys = [g for g in geom.geoms if isinstance(g, (Polygon, MultiPolygon))]

        if len(polys) == 0:
            return None

        if len(polys) == 1:
            return polys[0]

        merged = []
        for p in polys:
            if isinstance(p, Polygon):
                merged.append(p)
            else:  # MultiPolygon
                merged.extend(p.geoms)

        return MultiPolygon(merged)

    return geom


def build_department_command_unit_population_mapping(
    departments_file,
    command_units_file,
    population_raster,
    department_id_col,
    command_unit_id_col,
):
    departments_file = Path(departments_file)
    command_units_file = Path(command_units_file)
    population_raster = Path(population_raster)

    # Read vector data
    departments = gpd.read_file(departments_file)
    command_units = gpd.read_file(command_units_file)

    command_units["Name"] = command_units["Name"].map(mapping)

    # Match raster CRS
    with rasterio.open(population_raster) as src:
        raster_crs = src.crs

    departments = departments.to_crs(raster_crs)
    command_units = command_units.to_crs(raster_crs)

    departments["geometry"] = departments.geometry.apply(to_multipolygon)

    # Keep only needed columns
    departments = departments[[department_id_col, "geometry"]].copy()
    command_units = command_units[[command_unit_id_col, "geometry"]].copy()

    # Optional but useful
    departments = departments[
        ~departments.geometry.is_empty & departments.geometry.notna()
    ]
    command_units = command_units[
        ~command_units.geometry.is_empty & command_units.geometry.notna()
    ]

    departments["geometry"] = departments.geometry.make_valid()
    command_units["geometry"] = command_units.geometry.make_valid()

    departments["geometry"] = departments.geometry.apply(keep_polygons)
    departments = departments[departments.geometry.notna()].copy()

    # 1. Total population per department
    dept_pop = exact_extract(
        str(population_raster),
        departments,
        ["sum"],
        output="pandas",
    )

    dept_totals = departments[[department_id_col]].copy()
    dept_totals["department_population"] = dept_pop["sum"].fillna(0).to_numpy()

    # 2. Intersect departments with command units
    intersections = gpd.overlay(
        departments,
        command_units,
        how="intersection",
        keep_geom_type=True,
    )

    intersections = intersections[
        ~intersections.geometry.is_empty & intersections.geometry.notna()
    ].copy()

    # 3. Population per department-command-unit intersection
    inter_pop = exact_extract(
        str(population_raster),
        intersections,
        ["sum"],
        output="pandas",
    )

    intersections["population"] = inter_pop["sum"].fillna(0).to_numpy()

    # 4. Aggregate in case overlay creates multiple pieces per combination
    grouped = intersections.groupby(
        [department_id_col, command_unit_id_col], as_index=False
    )["population"].sum()

    # 5. Wide mapping: one row per department, one column per command unit
    population_mapping = grouped.pivot(
        index=department_id_col,
        columns=command_unit_id_col,
        values="population",
    ).fillna(0)

    # Add total department population
    population_mapping = population_mapping.merge(
        dept_totals.set_index(department_id_col),
        left_index=True,
        right_index=True,
        how="right",
    ).fillna(0)

    command_unit_cols = [
        col for col in population_mapping.columns if col != "department_population"
    ]

    # 6. Other population: people in department but outside all command units
    population_mapping["other"] = population_mapping[
        "department_population"
    ] - population_mapping[command_unit_cols].sum(axis=1)

    # Avoid tiny negative values from raster/vector precision
    population_mapping["other"] = population_mapping["other"].clip(lower=0)

    # 7. Convert to percentages
    percentage_mapping = population_mapping.copy()

    denominator = percentage_mapping["department_population"].replace(0, pd.NA)

    for col in command_unit_cols + ["other"]:
        percentage_mapping[col] = percentage_mapping[col] / denominator

    percentage_mapping = percentage_mapping.fillna(0)

    # Optional: remove raw total from percentage table
    percentage_mapping = percentage_mapping.drop(columns=["department_population"])

    return population_mapping.reset_index(), percentage_mapping.reset_index()

In [14]:
pop_file = src_dir / "egy_rural_population.tif"
gov_file = src_dir / "Governorates.shp"
com_file = src_dir / "Final2_Command_Area.shp"

In [21]:
population_mapping, percentage_mapping = (
    build_department_command_unit_population_mapping(
        departments_file=gov_file,
        command_units_file=com_file,
        population_raster=pop_file,
        department_id_col="NAME1_",
        command_unit_id_col="Name",
    )
)
percentage_mapping = percentage_mapping.set_index("NAME1_")
percentage_mapping.to_csv(src_dir.parent / "Result/population_density_matrix.csv")
merged.to_csv(src_dir.parent / "Result/ind_survey_data_gov_assigned.csv")

## 3. Assign survey data to command units
Using the population distribution matrix, we can randomly assign the survey points to a command unit based on the governorate available in that survey point. If a survey point was assigned to the command unit 'other', we filter it out of the data. We also merge the survey data with the agricultural data, using the assigned command unit.

In [17]:
governorates = merged[merged["NAME1_"].notna()]["NAME1_"].unique()

rng = np.random.default_rng(seed=42)

merged["command_unit"] = None
command_units = percentage_mapping.columns

for gov, idx in merged.groupby("NAME1_").groups.items():

    probs = percentage_mapping.loc[gov].to_numpy()

    assignments = rng.choice(
        command_units,
        size=len(idx),
        p=probs,
    )

    merged.loc[idx, "command_unit"] = assignments

merged.loc[merged["command_unit"] == "other", "command_unit"] = None

In [18]:
merged_nona = merged[merged["command_unit"].notna()]
merged_nona["occhd_label"].value_counts()

occhd_label
Skilled agricultural/fishery    9891
Elementary occupations           305
Managers                         248
Plant/machine operators          186
Technicians                      156
Craft workers                    148
Service/sales                    122
Clerks                            64
Professionals                     38
Name: count, dtype: int64

In [19]:
excel_df = pd.read_excel(excel_path, sheet_name="command_unit")
final_merged = merged_nona.merge(
    excel_df, left_on="command_unit", right_on="area", suffixes=("", "_spatial")
)
final_merged.to_csv(
    src_dir.parent / "Result/ind_survey_data_random_command_unit_assignment.csv"
)

In [20]:
final_merged

,country,year,round,dtype,caseser,pnum,pweight,reg,rururb,age,...,"drought_cultivation_area_Rice, paddy",salinity_production_Sugar cane,salinity_cultivation_area_Sugar cane,drought_production_Sugar cane,drought_cultivation_area_Sugar cane,salinity_production_Alfalfa for forage,salinity_cultivation_area_Alfalfa for forage,drought_production_Alfalfa for forage,drought_cultivation_area_Alfalfa for forage,producer_price
0,818,2022,3,1,2.302223e+07,1,315.016815,818023.0,0,25,...,0.000000,0,3854.216797,0,3854.216797,0.000000,0.000000,0.000000,0.000000,5.115124e+06
1,818,2022,4,1,1.503302e+08,2,343.413666,818015.0,0,57,...,1945.940552,0,0.000016,0,0.000016,10.561677,573.061035,10.561685,573.061035,8.642951e+06
2,818,2022,4,1,1.503302e+08,1,369.393005,818015.0,0,59,...,1945.940552,0,0.000016,0,0.000016,10.561677,573.061035,10.561685,573.061035,8.642951e+06
3,818,2022,4,1,1.510113e+08,1,369.393005,818015.0,0,64,...,1945.940552,0,0.000016,0,0.000016,10.561677,573.061035,10.561685,573.061035,8.642951e+06
4,818,2022,2,1,1.708503e+08,2,371.416504,818017.0,0,40,...,0.000000,0,983.164734,0,983.164734,5.498882,298.779907,6.050863,298.779907,5.902468e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12015,818,2022,2,1,2.906233e+11,1,214.657898,818029.0,0,50,...,2.038150,0,1119.076172,0,1119.076172,1.716374,143.115738,1.874078,143.115738,2.980832e+07
12016,818,2022,4,1,2.906233e+11,1,230.405182,818029.0,0,34,...,2.038150,0,1119.076172,0,1119.076172,1.716374,143.115738,1.874078,143.115738,2.980832e+07
12017,818,2022,4,1,2.906233e+11,1,230.405182,818029.0,0,43,...,2.038150,0,1119.076172,0,1119.076172,1.716374,143.115738,1.874078,143.115738,2.980832e+07
12018,818,2022,3,1,2.906233e+11,1,212.711319,818029.0,0,35,...,507.985718,0,6.878588,0,6.878588,0.507020,12.891274,0.507051,12.891274,1.686509e+07
